# LeetCode 1976 — Number of Ways to Arrive at Destination bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-arrive-at-destination/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Graph, Topological Sort, Shortest Path

Skeleton LeetCode:

```python
class Solution:
    def countPaths(self, n: int, roads: List[List[int]]) -> int:
        ...
```

Đếm số **đường đi ngắn nhất** từ `0` tới `n − 1`. Notebook đi từ **liệt kê mọi đường đi đơn** (vét cạn) → **Dijkstra mảng `O(n²)`** kèm đếm số cách → **Dijkstra với heap** (bỏ phần tử cũ!) → **hai pha**: tính khoảng cách rồi đếm trên **DAG đường ngắn nhất** theo thứ tự khoảng cách tăng dần.

## 1. Mục tiêu học

1. Ôn Dijkstra cho đồ thị trọng số **dương**, cả bản mảng `O(n²)` (đồ thị dày) và bản heap `O(E log V)`.
2. Mở rộng Dijkstra để **đếm số đường ngắn nhất**: `ways[v]` cập nhật khi `dist` **giảm** (gán lại) hoặc **bằng** (cộng thêm).
3. Hiểu "DAG đường ngắn nhất": cạnh `u → v` thuộc DAG ⇔ `dist[u] + w = dist[v]`; đếm đường trên DAG theo thứ tự `dist` tăng.
4. Tránh lỗi kinh điển: xử lý lại phần tử **cũ** trong heap gây đếm trùng; quên modulo; nhầm `<` với `≤`.
5. Kiểm thử bằng liệt kê đường đi đơn trên đồ thị nhỏ ngẫu nhiên, seed cố định.

## 2. Đề bài & ví dụ

Thành phố có `n` giao lộ đánh số `0..n−1`, các đường **hai chiều** giữa một số cặp giao lộ; đồ thị **liên thông**, giữa hai giao lộ có tối đa một đường. `roads[i] = [uᵢ, vᵢ, timeᵢ]`: đường nối `uᵢ`–`vᵢ` mất `timeᵢ` phút.

Trả về **số cách** đi từ giao lộ `0` tới `n − 1` trong **thời gian ngắn nhất**, modulo `10⁹ + 7`.

**Ví dụ 1:** `n = 7`, `roads = [[0,6,7],[0,1,2],[1,2,3],[1,3,3],[6,3,3],[3,5,1],[6,5,1],[2,5,1],[0,4,5],[4,6,2]]` → `4`. Thời gian ngắn nhất là 7: `0→6`, `0→4→6`, `0→1→2→5→6`, `0→1→3→5→6`.

**Ví dụ 2:** `n = 2`, `roads = [[1,0,10]]` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 200`, `n − 1 ≤ roads.length ≤ n(n−1)/2`, `1 ≤ timeᵢ ≤ 10⁹`, `uᵢ ≠ vᵢ`, không có cạnh lặp, đồ thị liên thông.

In [1]:
import heapq
import random
import time
from collections import defaultdict
from typing import List

MOD = 10**9 + 7
INF = float('inf')
examples = [
    (7, [[0, 6, 7], [0, 1, 2], [1, 2, 3], [1, 3, 3], [6, 3, 3], [3, 5, 1], [6, 5, 1], [2, 5, 1], [0, 4, 5], [4, 6, 2]], 4),
    (2, [[1, 0, 10]], 1),
]

def build_adj(n, roads):
    g = defaultdict(list)
    for u, v, t in roads:
        g[u].append((v, t)); g[v].append((u, t))
    return g

g = build_adj(*examples[0][:2])
for u in range(7):
    print(u, "->", sorted(g[u]))

0 -> [(1, 2), (4, 5), (6, 7)]
1 -> [(0, 2), (2, 3), (3, 3)]
2 -> [(1, 3), (5, 1)]
3 -> [(1, 3), (5, 1), (6, 3)]
4 -> [(0, 5), (6, 2)]
5 -> [(2, 1), (3, 1), (6, 1)]
6 -> [(0, 7), (3, 3), (4, 2), (5, 1)]


## 3. Kiến thức nền

### 3.1 Dijkstra với heap

Khởi tạo `dist[0] = 0`; heap chứa `(dist, đỉnh)`. Mỗi lần lấy đỉnh có `dist` nhỏ nhất **chưa chốt**; nới lỏng các cạnh kề. Với trọng số dương, lần đầu lấy một đỉnh ra là lúc `dist` của nó đã **chính xác**.

In [2]:
def dijkstra(n, roads, src=0):
    g = build_adj(n, roads)
    dist = [INF] * n; dist[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                      # phần tử cũ
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(pq, (dist[v], v))
    return dist

print("dist từ 0:", dijkstra(*examples[0][:2]))

dist từ 0: [0, 2, 5, 5, 5, 6, 7]


### 3.2 Đếm đường trên DAG theo thứ tự topo

Nếu đồ thị có hướng không chu trình, số đường từ `s` tới `v` là `ways[v] = Σ ways[u]` trên mọi cạnh `u → v`, tính theo thứ tự topo.

In [3]:
dag = {0: [1, 2], 1: [3], 2: [3], 3: []}
ways = {0: 1, 1: 0, 2: 0, 3: 0}
for u in [0, 1, 2, 3]:               # thứ tự topo
    for v in dag[u]:
        ways[v] += ways[u]
print(ways)

{0: 1, 1: 1, 2: 1, 3: 2}


### 3.3 Modulo khi cộng dồn

Số đường có thể bùng nổ theo cấp số nhân (ví dụ đồ thị "bậc thang" kim cương) — luôn `% MOD` sau mỗi phép cộng.

In [4]:
# chuỗi k "kim cương" song song: số đường ngắn nhất = 2^k
k = 70
print("2^70 =", 2**k, "-> mod:", pow(2, k, MOD))

2^70 = 1180591620717411303424 -> mod: 270016253


### 3.4 Ma trận kề cho đồ thị dày

Khi `E ≈ n²/2`, Dijkstra mảng `O(n²)` (không heap) gọn và nhanh tương đương. Ma trận `g[u][v] = w` hoặc `INF`.

In [5]:
n, roads = examples[0][:2]
mat = [[INF] * n for _ in range(n)]
for u, v, t in roads:
    mat[u][v] = mat[v][u] = t
print(mat[0])

[inf, 2, inf, inf, 5, inf, 7]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1, `dist = [0, 2, 5, 5, 5, 6, 7]`. Cạnh `u → v` "nằm trên một đường ngắn nhất" khi `dist[u] + w(u, v) = dist[v]`:
- `0→1` (0+2=2), `1→2` (2+3=5), `1→3` (2+3=5), `0→4` (0+5=5), `2→5` (5+1=6), `3→5` (5+1=6), `5→6` (6+1=7), `4→6` (5+2=7), `0→6` (0+7=7). (`3→6`: 5+3=8 ≠ 7 ⇒ không.)

Đếm theo thứ tự `dist` tăng: `ways[0]=1`, `ways[1]=1`, `ways[2]=ways[3]=1`, `ways[4]=1`, `ways[5]=ways[2]+ways[3]=2`, `ways[6]=ways[5]+ways[4]+ways[0]=2+1+1=4` ✔.

In [6]:
n, roads, _ = examples[0]
dist = dijkstra(n, roads)
tight = [(u, v, t) for a, b, t in roads for u, v in ((a, b), (b, a)) if dist[u] + t == dist[v]]
print("dist:", dist)
print("cạnh trên DAG đường ngắn nhất:", sorted(tight))

dist: [0, 2, 5, 5, 5, 6, 7]
cạnh trên DAG đường ngắn nhất: [(0, 1, 2), (0, 4, 5), (0, 6, 7), (1, 2, 3), (1, 3, 3), (2, 5, 1), (3, 5, 1), (4, 6, 2), (5, 6, 1)]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường đi đơn

DFS từ `0`, không thăm lại đỉnh, ghi nhận độ dài khi tới `n − 1`. Đường ngắn nhất (trọng số dương) luôn là **đường đơn** nên liệt kê đường đơn là đủ.

In [7]:
def count_paths_brute(n, roads):
    if n == 1:
        return 1
    g = build_adj(n, roads)
    best, cnt = INF, 0
    seen = [False] * n
    def dfs(u, d):
        nonlocal best, cnt
        if d > best:
            return                          # cắt nhánh: đã dài hơn đường tốt nhất
        if u == n - 1:
            if d < best:
                best, cnt = d, 1
            else:
                cnt += 1
            return
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True; dfs(v, d + w); seen[v] = False
    seen[0] = True
    dfs(0, 0)
    return cnt % MOD

for n_, roads_, exp in examples:
    print(count_paths_brute(n_, roads_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp:** số đường đơn có thể tới `O(n!)` — chỉ cho `n ≤ 9`.

**Nút thắt:** nhiều đường chung đoạn đầu được duyệt lại. Nếu biết `dist[u]` (độ dài ngắn nhất tới `u`) và `ways[u]`, ta không cần nhớ **cả đường**.

## 6. Ý tưởng thuật toán / bước đột phá

**Dijkstra + đếm:** `ways[0] = 1`. Khi nới lỏng cạnh `u → v` (u vừa được chốt) với `nd = dist[u] + w`:
- **Nếu** `nd < dist[v]` **thì** tìm được đường ngắn hơn: `dist[v] = nd`, `ways[v] = ways[u]` (bỏ hết cách cũ).
- **Nếu** `nd == dist[v]` **thì** thêm cách: `ways[v] += ways[u]`.

**Bất biến:** khi một đỉnh `u` được **chốt** (lấy ra lần đầu với `d == dist[u]`), mọi đỉnh `x` có `dist[x] < dist[u]` đã được chốt và đã nới lỏng ⇒ `ways[u]` đã **đầy đủ**. Vì vậy khi `u` nới lỏng các cạnh, giá trị `ways[u]` truyền đi là cuối cùng.

**Hai pha (tương đương, dễ kiểm):** tính `dist` bằng Dijkstra; sắp đỉnh theo `dist`; duyệt tăng dần, với mỗi cạnh "khít" `dist[u] + w = dist[v]` cộng `ways[u]` vào `ways[v]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dijkstra mảng `O(n²)` (đồ thị dày)

In [8]:
def count_paths_dense(n, roads):
    g = [[INF] * n for _ in range(n)]
    for u, v, t in roads:
        g[u][v] = g[v][u] = t
    dist = [INF] * n; dist[0] = 0
    ways = [0] * n; ways[0] = 1
    done = [False] * n
    for _ in range(n):
        u = -1
        for j in range(n):
            if not done[j] and (u == -1 or dist[j] < dist[u]):
                u = j
        done[u] = True
        for v in range(n):
            if done[v] or g[u][v] == INF:
                continue
            nd = dist[u] + g[u][v]
            if nd < dist[v]:
                dist[v], ways[v] = nd, ways[u]
            elif nd == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1] % MOD

for n_, roads_, exp in examples:
    print(count_paths_dense(n_, roads_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n²)` thời gian, `O(n²)` bộ nhớ (ma trận). Với `n = 200` rất nhanh.

### 7.2 Cách 3 — Dijkstra với heap (lời giải chính)

In [9]:
def count_paths_heap(n, roads):
    g = build_adj(n, roads)
    dist = [INF] * n; dist[0] = 0
    ways = [0] * n; ways[0] = 1
    pq = [(0, 0)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                                  # BỎ phần tử cũ — tránh đếm trùng
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v], ways[v] = nd, ways[u]
                heapq.heappush(pq, (nd, v))
            elif nd == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1] % MOD

for n_, roads_, exp in examples:
    print(count_paths_heap(n_, roads_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(E log E)` thời gian, `O(n + E)` bộ nhớ.

**Chi tiết tinh tế:** khi `u` được chốt, mọi hàng xóm `v` đã chốt trước đó có `dist[v] ≤ dist[u] < dist[u] + w`, nên nhánh `==` không bao giờ cộng vào một đỉnh đã chốt. Điều quan trọng là **mỗi đỉnh chỉ "phát" `ways` một lần** — nhờ dòng `continue` bỏ phần tử cũ.

### 7.3 Cách 4 — Hai pha: khoảng cách rồi đếm trên DAG

In [10]:
def count_paths_two_phase(n, roads):
    dist = dijkstra(n, roads)
    g = build_adj(n, roads)
    order = sorted(range(n), key=lambda x: dist[x])
    ways = [0] * n; ways[0] = 1
    for u in order:
        for v, w in g[u]:
            if dist[u] + w == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1]

for n_, roads_, exp in examples:
    print(count_paths_two_phase(n_, roads_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


Thứ tự `dist` tăng là một **thứ tự topo** của DAG (mọi cạnh khít đi từ `dist` nhỏ sang `dist` lớn vì `w > 0`). **Độ phức tạp:** `O(E log E + n log n + E)`.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 — in mỗi lần chốt đỉnh và các cập nhật:

In [11]:
def trace_heap(n, roads):
    g = build_adj(n, roads)
    dist = [INF] * n; dist[0] = 0
    ways = [0] * n; ways[0] = 1
    pq = [(0, 0)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            print(f"  lấy ({d}, {u}): cũ (dist[{u}] = {dist[u]}) -> bỏ qua")
            continue
        print(f"  chốt đỉnh {u}: dist = {d}, ways = {ways[u]}")
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v], ways[v] = nd, ways[u]; heapq.heappush(pq, (nd, v))
                print(f"     {u}->{v} (w={w}): {nd} < cũ -> dist[{v}]={nd}, ways[{v}]={ways[v]}")
            elif nd == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
                print(f"     {u}->{v} (w={w}): {nd} == dist -> ways[{v}] += {ways[u]} = {ways[v]}")
    print("  dist =", dist, "| ways =", ways)
    return ways[n - 1]

trace_heap(*examples[0][:2])

  chốt đỉnh 0: dist = 0, ways = 1
     0->6 (w=7): 7 < cũ -> dist[6]=7, ways[6]=1
     0->1 (w=2): 2 < cũ -> dist[1]=2, ways[1]=1
     0->4 (w=5): 5 < cũ -> dist[4]=5, ways[4]=1
  chốt đỉnh 1: dist = 2, ways = 1
     1->2 (w=3): 5 < cũ -> dist[2]=5, ways[2]=1
     1->3 (w=3): 5 < cũ -> dist[3]=5, ways[3]=1
  chốt đỉnh 2: dist = 5, ways = 1
     2->5 (w=1): 6 < cũ -> dist[5]=6, ways[5]=1
  chốt đỉnh 3: dist = 5, ways = 1
     3->5 (w=1): 6 == dist -> ways[5] += 1 = 2
  chốt đỉnh 4: dist = 5, ways = 1
     4->6 (w=2): 7 == dist -> ways[6] += 1 = 2
  chốt đỉnh 5: dist = 6, ways = 2
     5->6 (w=1): 7 == dist -> ways[6] += 2 = 4
  chốt đỉnh 6: dist = 7, ways = 4
  dist = [0, 2, 5, 5, 5, 6, 7] | ways = [1, 1, 1, 1, 1, 2, 4]


4

Bảng tóm tắt (thứ tự chốt):

| đỉnh chốt | dist | ways | tác động chính |
|---|---|---|---|
| 0 | 0 | 1 | dist[6]=7, dist[1]=2, dist[4]=5 |
| 1 | 2 | 1 | dist[2]=5, dist[3]=5 |
| 2, 3, 4 | 5 | 1 | dist[5]=6 (từ 2), ways[5]+=1 (từ 3); 4→6: 7 == 7 ⇒ ways[6]=2 |
| 5 | 6 | 2 | 5→6: 7 == 7 ⇒ ways[6] = 4 |
| 6 | 7 | 4 | |

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def countPaths(self, n: int, roads: List[List[int]]) -> int:
        MOD = 10**9 + 7
        g = [[] for _ in range(n)]
        for u, v, t in roads:
            g[u].append((v, t))
            g[v].append((u, t))
        dist = [float('inf')] * n
        ways = [0] * n
        dist[0], ways[0] = 0, 1
        pq = [(0, 0)]
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]:
                continue
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v], ways[v] = nd, ways[u]
                    heapq.heappush(pq, (nd, v))
                elif nd == dist[v]:
                    ways[v] = (ways[v] + ways[u]) % MOD
        return ways[n - 1] % MOD

for n_, roads_, exp in examples:
    assert Solution().countPaths(n_, roads_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `D(v)` là khoảng cách ngắn nhất thật, `W(v)` là số đường ngắn nhất thật tới `v`.

**DAG đường ngắn nhất.** Một đường `0 = x₀ → … → x_k = v` là ngắn nhất ⇔ mọi cạnh đều "khít": `D(xᵢ) + w = D(xᵢ₊₁)`. Do `w > 0`, cạnh khít luôn đi từ `D` nhỏ sang `D` lớn ⇒ đồ thị các cạnh khít là DAG, và `W(v) = Σ_{u: cạnh khít u→v} W(u)`.

**Hai pha (cách 4).** Duyệt đỉnh theo `D` tăng là thứ tự topo của DAG ⇒ khi xử lý `u`, `ways[u] = W(u)` đã đầy đủ; cộng vào mọi `v` khít ⇒ đúng.

**Dijkstra đếm (cách 2, 3).** Bất biến: khi `u` được chốt, `dist[u] = D(u)` và `ways[u] = W(u)`. Mọi tiền nhiệm khít `x` của `u` có `D(x) < D(u)` nên đã được chốt **trước** `u` và lúc đó đã cộng (hoặc gán) `ways[x]` vào `ways[u]` — các lần "gán" (khi `dist` giảm) chỉ xảy ra trước khi `dist[u]` đạt `D(u)`, xóa đi các đóng góp từ đường dài hơn. Phần tử cũ trong heap bị bỏ qua nên mỗi `u` phát `ways[u]` đúng một lần.

## 11. Phân tích Time / Space Complexity

Gọi `V = n`, `E = len(roads)` (`≤ n(n−1)/2 ≈ 2·10⁴`).

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường đơn | `O(V!)` | `O(V)` |
| 2. Dijkstra mảng | `O(V²)` | `O(V²)` |
| 3. Dijkstra heap | `O(E log E)` | `O(V + E)` |
| 4. Hai pha | `O(E log E + V log V)` | `O(V + E)` |

In [13]:
def random_graph(n, rnd, extra_p=0.3, wmax=10**9):
    roads, have = [], set()
    perm = list(range(n)); rnd.shuffle(perm)
    for i in range(1, n):                         # cây khung bảo đảm liên thông
        u, v = perm[i], perm[rnd.randrange(i)]
        roads.append([u, v, rnd.randint(1, wmax)]); have.add((min(u, v), max(u, v)))
    for u in range(n):
        for v in range(u + 1, n):
            if (u, v) not in have and rnd.random() < extra_p:
                roads.append([u, v, rnd.randint(1, wmax)])
    return roads

rnd = random.Random(1976)
n_big = 200
roads_big = random_graph(n_big, rnd, extra_p=1.0, wmax=3)          # đồ thị đầy đủ, trọng số nhỏ -> nhiều đường bằng nhau
for name, fn in [("dense", count_paths_dense), ("heap", count_paths_heap), ("two_phase", count_paths_two_phase), ("Solution", Solution().countPaths)]:
    t0 = time.perf_counter(); r = fn(n_big, roads_big); dt = time.perf_counter() - t0
    print(f"{name:<10} n=200, E={len(roads_big)}: {r:>10}  {dt * 1000:7.1f} ms")

dense      n=200, E=19900:         25      4.3 ms
heap       n=200, E=19900:         25      6.9 ms
two_phase  n=200, E=19900:         25      8.9 ms
Solution   n=200, E=19900:         25     20.5 ms


## 12. Bộ test

In [14]:
ALL = {
    "dense": count_paths_dense,
    "heap": count_paths_heap,
    "two_phase": count_paths_two_phase,
    "Solution": lambda n, r: Solution().countPaths(n, r),
}

def check(n, roads, exp=None, brute=True):
    ref = count_paths_brute(n, roads) if brute else count_paths_two_phase(n, roads)
    for name, fn in ALL.items():
        got = fn(n, [r[:] for r in roads])
        assert got == ref, (name, n, roads, got, ref)
    if exp is not None:
        assert ref == exp, (ref, exp)
    return ref

for n_, roads_, exp in examples:
    check(n_, roads_, exp)
print("Ví dụ đề: OK")

check(1, [], 1)                                                    # chỉ một giao lộ
check(3, [[0, 1, 1], [1, 2, 1], [0, 2, 2]], 2)                     # hai đường bằng nhau
check(4, [[0, 1, 1], [0, 2, 1], [1, 3, 1], [2, 3, 1]], 2)          # kim cương
check(4, [[0, 1, 1], [0, 2, 1], [1, 3, 1], [2, 3, 2]], 1)
print("Edge case: OK")

# chuỗi k kim cương: số đường = 2^k mod MOD (kiểm modulo)
k = 40
roads_chain, node = [], 0
for _ in range(k):
    a, b, c = node + 1, node + 2, node + 3
    roads_chain += [[node, a, 1], [node, b, 1], [a, c, 1], [b, c, 1]]
    node = c
check(node + 1, roads_chain, pow(2, k, MOD), brute=False)
print("Chuỗi 40 kim cương: 2^40 mod MOD: OK")

Ví dụ đề: OK
Edge case: OK
Chuỗi 40 kim cương: 2^40 mod MOD: OK


In [15]:
rnd = random.Random(2002)
for _ in range(600):
    n_ = rnd.randint(2, 8)
    roads_ = random_graph(n_, rnd, extra_p=rnd.choice([0.2, 0.5, 0.9]), wmax=rnd.choice([1, 3, 10]))
    check(n_, roads_)
print("600 đồ thị nhỏ ngẫu nhiên so với liệt kê đường đơn: OK")

for _ in range(20):
    n_ = rnd.randint(2, 200)
    check(n_, random_graph(n_, rnd, extra_p=0.2, wmax=rnd.choice([2, 10**9])), brute=False)
print("20 đồ thị cỡ lớn: các cách khớp nhau: OK")

600 đồ thị nhỏ ngẫu nhiên so với liệt kê đường đơn: OK


20 đồ thị cỡ lớn: các cách khớp nhau: OK


Các edge case ở ô trên là **tự dựng**; kỳ vọng đã được xác nhận bằng liệt kê đường đơn (đối chiếu lại đề gốc nếu muốn).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không bỏ phần tử cũ trong heap (đếm trùng)

In [16]:
def count_paths_bug_stale(n, roads):
    g = build_adj(n, roads)
    dist = [INF] * n; dist[0] = 0
    ways = [0] * n; ways[0] = 1
    pq = [(0, 0)]
    while pq:
        d, u = heapq.heappop(pq)                # SAI: không kiểm d > dist[u]
        for v, w in g[u]:
            nd = dist[u] + w
            if nd < dist[v]:
                dist[v], ways[v] = nd, ways[u]; heapq.heappush(pq, (nd, v))
            elif nd == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1]

rnd = random.Random(1)
found = None
while found is None:
    n_ = rnd.randint(3, 7); roads_ = random_graph(n_, rnd, extra_p=0.7, wmax=3)
    if count_paths_bug_stale(n_, roads_) != Solution().countPaths(n_, roads_):
        found = (n_, roads_)
print("phản ví dụ:", found)
print("   không bỏ phần tử cũ:", count_paths_bug_stale(*found), "| đúng:", Solution().countPaths(*found))

phản ví dụ: (5, [[1, 2, 1], [3, 2, 1], [4, 2, 1], [0, 3, 1], [0, 2, 3], [0, 4, 3], [3, 4, 2]])
   không bỏ phần tử cũ: 4 | đúng: 3


**Vì sao sai:** một đỉnh có thể nằm trong heap nhiều lần (mỗi lần `dist` giảm). Nếu xử lý cả bản cũ, nó "phát" `ways` **nhiều lần** sang hàng xóm ⇒ đếm trùng. **Sửa:** `if d > dist[u]: continue` (hoặc mảng `done`).

### Lỗi 2 — Khi tìm được đường ngắn hơn thì **cộng** thay vì **gán** `ways`

In [17]:
def count_paths_bug_add(n, roads):
    g = build_adj(n, roads)
    dist = [INF] * n; dist[0] = 0
    ways = [0] * n; ways[0] = 1
    pq = [(0, 0)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd; ways[v] = (ways[v] + ways[u]) % MOD      # SAI: phải gán lại
                heapq.heappush(pq, (nd, v))
            elif nd == dist[v]:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1]

n_, roads_ = 3, [[0, 2, 5], [0, 1, 1], [1, 2, 1]]
print("cộng thay vì gán:", count_paths_bug_add(n_, roads_), "| đúng:", Solution().countPaths(n_, roads_))

cộng thay vì gán: 2 | đúng: 1


**Vì sao sai:** cách đi `0→2` (độ dài 5) đã được đếm, sau đó tìm ra đường ngắn hơn `0→1→2` (độ dài 2) — các cách cũ **không còn** là ngắn nhất, phải **xóa**. **Sửa:** `ways[v] = ways[u]` khi `nd < dist[v]`.

### Lỗi 3 — Dùng BFS (bỏ qua trọng số) hoặc đếm mọi đường

In [18]:
from collections import deque as _dq
def count_paths_bug_bfs(n, roads):
    g = build_adj(n, roads)
    dist = [-1] * n; dist[0] = 0; ways = [0] * n; ways[0] = 1
    q = _dq([0])
    while q:
        u = q.popleft()
        for v, _ in g[u]:                            # SAI: coi mọi cạnh dài 1
            if dist[v] == -1:
                dist[v] = dist[u] + 1; ways[v] = ways[u]; q.append(v)
            elif dist[v] == dist[u] + 1:
                ways[v] += ways[u]
    return ways[n - 1] % MOD

n_, roads_, exp = examples[0]
print("BFS:", count_paths_bug_bfs(n_, roads_), "| đúng:", Solution().countPaths(n_, roads_))

BFS: 1 | đúng: 4


**Vì sao sai:** "ngắn nhất" theo **thời gian** (tổng trọng số), không theo số cạnh. **Sửa:** Dijkstra.

### Lỗi 4 — Quên modulo

In [19]:
def count_paths_bug_nomod(n, roads):
    dist = dijkstra(n, roads); g = build_adj(n, roads)
    ways = [0] * n; ways[0] = 1
    for u in sorted(range(n), key=lambda x: dist[x]):
        for v, w in g[u]:
            if dist[u] + w == dist[v]:
                ways[v] += ways[u]                  # SAI: không lấy mod
    return ways[n - 1]

print("không mod (40 kim cương):", count_paths_bug_nomod(node + 1, roads_chain), "| có mod:", Solution().countPaths(node + 1, roads_chain))

không mod (40 kim cương): 1099511627776 | có mod: 511620083


**Vì sao sai:** đề yêu cầu trả về modulo `10⁹ + 7`; số đường có thể tới `2^{n/3}`. Python không tràn số nhưng kết quả **sai yêu cầu** (và chậm với số rất lớn); C++/Java thì tràn. **Sửa:** `% MOD` mỗi lần cộng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số đường **ngắn nhất**" ⇒ Dijkstra (trọng số dương) / BFS (không trọng số) **kèm mảng `ways`**: `<` thì gán, `==` thì cộng.
- Dijkstra heap trong Python: **luôn** có dòng bỏ phần tử cũ.
- Đồ thị dày (`E ≈ V²/2`, `V ≤ 500`) ⇒ Dijkstra mảng `O(V²)` cũng tốt.
- Phiên bản hai pha (dist rồi DAG) dễ kiểm và dễ mở rộng (đếm đường ngắn nhất đi qua một cạnh, v.v.).
- Kết quả lớn ⇒ modulo mỗi bước.

## 15. Biến thể của bài

1. **Không trọng số:** BFS + `ways`.
2. **Đếm đường ngắn nhất đi qua cạnh `e`:** `ways_from_0[u] · ways_to_end[v]` nếu `e = (u, v)` khít theo cả hai chiều khoảng cách.
3. **LeetCode 1786 — Number of Restricted Paths:** đếm đường mà khoảng cách tới đích giảm nghiêm ngặt — DP trên thứ tự dist.
4. **Đường ngắn thứ hai / k đường ngắn nhất:** Dijkstra giữ `k` giá trị mỗi đỉnh.
5. **Trọng số có thể bằng 0:** cạnh 0 có thể tạo chu trình "khít" ⇒ số đường có thể vô hạn; cần xử lý thành phần trọng số 0.

In [20]:
def count_paths_through_edge(n, roads):
    """Biến thể 2: với mỗi cạnh, số đường ngắn nhất 0 -> n-1 đi qua nó."""
    d0 = dijkstra(n, roads); dn = dijkstra(n, roads, src=n - 1)
    g = build_adj(n, roads)
    f = [0] * n; f[0] = 1
    for u in sorted(range(n), key=lambda x: d0[x]):
        for v, w in g[u]:
            if d0[u] + w == d0[v]: f[v] += f[u]
    b = [0] * n; b[n - 1] = 1
    for u in sorted(range(n), key=lambda x: dn[x]):
        for v, w in g[u]:
            if dn[u] + w == dn[v]: b[v] += b[u]
    total = d0[n - 1]; res = {}
    for u, v, w in roads:
        c = 0
        if d0[u] + w + dn[v] == total: c += f[u] * b[v]
        if d0[v] + w + dn[u] == total: c += f[v] * b[u]
        res[(u, v)] = c
    return res

r = count_paths_through_edge(*examples[0][:2])
print({e: c for e, c in r.items() if c})

{(0, 6): 1, (0, 1): 2, (1, 2): 1, (1, 3): 1, (3, 5): 1, (6, 5): 2, (2, 5): 1, (0, 4): 1, (4, 6): 1}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | DFS mọi đường đơn | `O(V!)` | `O(V)` | kiểm thử `V ≤ 8` |
| 2. Dijkstra mảng | chọn min tuyến tính | `O(V²)` | `O(V²)` | đồ thị dày |
| 3. Dijkstra heap | `<` gán, `==` cộng | `O(E log E)` | `O(V+E)` | **nộp bài** |
| 4. Hai pha | dist rồi đếm trên DAG | `O(E log E)` | `O(V+E)` | dễ kiểm, dễ mở rộng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().countPaths(4, [[0,1,2],[0,2,2],[1,3,2],[2,3,2],[0,3,4]])`?

**Bài 2 (tính tay):** Với `n = 5`, `roads = [[0,1,1],[0,2,1],[1,2,1],[1,3,1],[2,3,1],[3,4,1],[2,4,2]]`, tính `dist` và `ways` của từng đỉnh theo thứ tự `dist`.

**Bài 3 (điền TODO):** Hoàn thiện `all_shortest_paths(n, roads)` **liệt kê** mọi đường ngắn nhất (danh sách các đỉnh) bằng DFS trên DAG khít (đồ thị nhỏ).

**Bài 4 (tự viết lại):** Viết `count_paths_unweighted(n, edges)` cho đồ thị **không trọng số** bằng BFS và kiểm bằng `Solution` với mọi trọng số = 1.

**Bài 5 (mini challenge):** Viết `count_restricted_paths(n, edges)` (LeetCode 1786): đếm đường `1 → n` (1-indexed) sao cho `distToLast` giảm nghiêm ngặt dọc đường.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def all_shortest_paths(n, roads):
    # TODO: dist = dijkstra; DFS từ 0 chỉ đi cạnh khít dist[u] + w == dist[v]
    pass

# Bài 4 — TODO
def count_paths_unweighted(n, edges):
    # TODO: BFS; ways[v] = ways[u] khi lần đầu; += khi cùng tầng
    pass

# Bài 5 — TODO
def count_restricted_paths(n, edges):
    # TODO: Dijkstra từ n; DP theo dist tăng: f[u] = sum f[v] với dist[v] < dist[u]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `dist[3] = 4`.
- Mức 2: Ba đường độ dài 4: `0→3`, `0→1→3`, `0→2→3`.
- Mức 3: Đáp án 3.

**Bài 2**
- Mức 1: `dist = [0, 1, 1, 2, 3]`.
- Mức 2: `ways[3] = ways[1] + ways[2] = 2`.
- Mức 3: `ways[4] = ways[3] + ways[2]` (vì `1 + 2 = 3`) `= 3`.

**Bài 3**
- Mức 1: DAG khít không có chu trình ⇒ DFS dừng.
- Mức 2: Số đường liệt kê phải bằng `Solution()`.
- Mức 3: Mỗi đường có tổng trọng số `= dist[n−1]`.

**Bài 4**
- Mức 1: BFS cho `dist` theo số cạnh.
- Mức 2: Cạnh khít: `dist[v] == dist[u] + 1`.
- Mức 3: So với `Solution` sau khi gán trọng số 1.

**Bài 5**
- Mức 1: Tính `d` từ đỉnh `n`.
- Mức 2: `f[n] = 1`; xử lý đỉnh theo `d` tăng.
- Mức 3: `f[u] = Σ f[v]` trên cạnh `u–v` với `d[v] < d[u]`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().countPaths(4, [[0, 1, 2], [0, 2, 2], [1, 3, 2], [2, 3, 2], [0, 3, 4]])
print("Bài 1:", r1)
assert r1 == 3

# Bài 2
n2, roads2 = 5, [[0, 1, 1], [0, 2, 1], [1, 2, 1], [1, 3, 1], [2, 3, 1], [3, 4, 1], [2, 4, 2]]
trace_heap(n2, roads2)
assert Solution().countPaths(n2, roads2) == 3 == count_paths_brute(n2, roads2)

# Bài 3
def all_shortest_paths_sol(n, roads):
    dist = dijkstra(n, roads); g = build_adj(n, roads)
    out = []
    def dfs(u, path):
        if u == n - 1:
            out.append(path[:]); return
        for v, w in g[u]:
            if dist[u] + w == dist[v]:
                path.append(v); dfs(v, path); path.pop()
    dfs(0, [0])
    return out

paths = all_shortest_paths_sol(*examples[0][:2])
print("Bài 3:", paths)
assert len(paths) == 4

Bài 1: 3
  chốt đỉnh 0: dist = 0, ways = 1
     0->1 (w=1): 1 < cũ -> dist[1]=1, ways[1]=1
     0->2 (w=1): 1 < cũ -> dist[2]=1, ways[2]=1
  chốt đỉnh 1: dist = 1, ways = 1
     1->3 (w=1): 2 < cũ -> dist[3]=2, ways[3]=1
  chốt đỉnh 2: dist = 1, ways = 1
     2->3 (w=1): 2 == dist -> ways[3] += 1 = 2
     2->4 (w=2): 3 < cũ -> dist[4]=3, ways[4]=1
  chốt đỉnh 3: dist = 2, ways = 2
     3->4 (w=1): 3 == dist -> ways[4] += 2 = 3
  chốt đỉnh 4: dist = 3, ways = 3
  dist = [0, 1, 1, 2, 3] | ways = [1, 1, 1, 2, 3]
Bài 3: [[0, 6], [0, 1, 2, 5, 6], [0, 1, 3, 5, 6], [0, 4, 6]]


In [23]:
# Bài 4
def count_paths_unweighted_sol(n, edges):
    g = defaultdict(list)
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    dist = [-1] * n; ways = [0] * n
    dist[0], ways[0] = 0, 1
    q = [0]; head = 0
    while head < len(q):
        u = q[head]; head += 1
        for v in g[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1; ways[v] = ways[u]; q.append(v)
            elif dist[v] == dist[u] + 1:
                ways[v] = (ways[v] + ways[u]) % MOD
    return ways[n - 1] % MOD

rnd = random.Random(4)
for _ in range(300):
    n_ = rnd.randint(2, 30)
    roads_ = random_graph(n_, rnd, extra_p=0.3, wmax=1)
    assert count_paths_unweighted_sol(n_, [(u, v) for u, v, _ in roads_]) == Solution().countPaths(n_, roads_)
print("Bài 4 OK")

# Bài 5
def count_restricted_paths_sol(n, edges):
    roads = [[u - 1, v - 1, w] for u, v, w in edges]
    d = dijkstra(n, roads, src=n - 1); g = build_adj(n, roads)
    f = [0] * n; f[n - 1] = 1
    for u in sorted(range(n), key=lambda x: d[x]):
        for v, w in g[u]:
            if d[v] < d[u]:
                f[u] = (f[u] + f[v]) % MOD
    return f[0]

print("Bài 5: 1786 ví dụ 1 ->", count_restricted_paths_sol(5, [[1, 2, 3], [1, 3, 3], [2, 3, 1], [1, 4, 2], [5, 2, 2], [3, 5, 1], [5, 4, 10]]))
assert count_restricted_paths_sol(5, [[1, 2, 3], [1, 3, 3], [2, 3, 1], [1, 4, 2], [5, 2, 2], [3, 5, 1], [5, 4, 10]]) == 3

Bài 4 OK
Bài 5: 1786 ví dụ 1 -> 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Dijkstra đếm số đường ngắn nhất:**

```python
dist = [inf] * n; ways = [0] * n
dist[s], ways[s] = 0, 1
pq = [(0, s)]
while pq:
    d, u = heappop(pq)
    if d > dist[u]: continue              # bỏ phần tử cũ
    for v, w in g[u]:
        nd = d + w
        if nd < dist[v]:
            dist[v], ways[v] = nd, ways[u]  # GÁN
            heappush(pq, (nd, v))
        elif nd == dist[v]:
            ways[v] = (ways[v] + ways[u]) % MOD   # CỘNG
```

**Checklist:**
- [ ] Bỏ qua phần tử heap cũ (`d > dist[u]`)?
- [ ] `<` thì **gán**, `==` thì **cộng**?
- [ ] Dùng Dijkstra (trọng số), không phải BFS?
- [ ] Modulo mỗi lần cộng?
- [ ] Đồ thị hai chiều: thêm cạnh cả hai hướng?
- [ ] `n = 1` trả `1`?
- [ ] Chữ ký đúng: `def countPaths(self, n: int, roads: List[List[int]]) -> int`?